# 😷 Face Mask Detection — Exploratory Data Analysis

This notebook explores the Kaggle Face Mask Detection dataset before training.

In [5]:
import sys
sys.path.insert(0, '../src')

import os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import seaborn as sns

import sys
import os

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))


from dataset import parse_annotation, LABEL_MAP, LABEL_NAMES

DATA_DIR        = Path('../data')
IMAGES_DIR      = DATA_DIR / 'images'
ANNOTATIONS_DIR = DATA_DIR / 'annotations'

print('Images:',      len(list(IMAGES_DIR.glob('*'))))
print('Annotations:', len(list(ANNOTATIONS_DIR.glob('*.xml'))))

ModuleNotFoundError: No module named 'dataset'

## 1 — Class Distribution

In [ ]:
label_counts = Counter()
for xml_path in ANNOTATIONS_DIR.glob('*.xml'):
    for obj in parse_annotation(str(xml_path)):
        label_counts[obj['label']] += 1

print(dict(label_counts))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c', '#f39c12']
bars   = ax.bar(label_counts.keys(), label_counts.values(), color=colors, edgecolor='white')
ax.bar_label(bars, padding=4, fontsize=12)
ax.set_title('Face annotations per class', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 2 — Sample images with bounding boxes

In [ ]:
COLOR_MAP = {'with_mask': '#2ecc71', 'without_mask': '#e74c3c', 'mask_weared_incorrect': '#f39c12'}

xml_files = sorted(ANNOTATIONS_DIR.glob('*.xml'))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax, xml_path in zip(axes.flat, xml_files):
    stem     = xml_path.stem
    img_path = IMAGES_DIR / f'{stem}.png'
    if not img_path.exists():
        img_path = IMAGES_DIR / f'{stem}.jpg'
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    for obj in parse_annotation(str(xml_path)):
        x, y    = obj['xmin'], obj['ymin']
        bw, bh  = obj['xmax'] - x, obj['ymax'] - y
        color   = COLOR_MAP.get(obj['label'], 'blue')
        rect    = patches.Rectangle((x, y), bw, bh, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 4, obj['label'], color=color, fontsize=7, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample images with annotations', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 3 — Bounding box size distribution

In [ ]:
widths, heights = [], []
for xml_path in ANNOTATIONS_DIR.glob('*.xml'):
    for obj in parse_annotation(str(xml_path)):
        widths.append(obj['xmax']  - obj['xmin'])
        heights.append(obj['ymax'] - obj['ymin'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(widths,  bins=40, color='steelblue',  edgecolor='white')
axes[0].set_title('Bounding box widths');  axes[0].set_xlabel('pixels')
axes[1].hist(heights, bins=40, color='salmon',     edgecolor='white')
axes[1].set_title('Bounding box heights'); axes[1].set_xlabel('pixels')
for ax in axes:
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Width  — mean: {np.mean(widths):.0f}px  std: {np.std(widths):.0f}px')
print(f'Height — mean: {np.mean(heights):.0f}px  std: {np.std(heights):.0f}px')

## 4 — After training: visualise predictions

In [ ]:
# Run this cell after training to visualise model predictions on the test set
import tensorflow as tf
from dataset import extract_faces, split_dataset

X, y = extract_faces(str(DATA_DIR))
_, _, X_test, _, _, y_test = split_dataset(X, y)

model = tf.keras.models.load_model('../models/mask_detector.h5')
preds = np.argmax(model.predict(X_test[:36]), axis=1)

fig, axes = plt.subplots(6, 6, figsize=(14, 14))
for ax, img, true, pred in zip(axes.flat, X_test, y_test, preds):
    ax.imshow(img)
    color = 'green' if true == pred else 'red'
    ax.set_title(f'T:{LABEL_NAMES[true][0]}  P:{LABEL_NAMES[pred][0]}', fontsize=8, color=color)
    ax.axis('off')
plt.suptitle('Test set predictions (green=correct, red=wrong)', fontsize=13)
plt.tight_layout()
plt.show()